In [11]:
import os
import pandas as pd
import numpy as np

In [12]:
#Define your base path
base_path = '/Users/vidhi/college baby/sem 5/ASER dataset/ASER HH compiled'

# List of years you want to process
years = [2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2016, 2018, 2022, 2024]

# A dictionary to temporarily hold the dataframes
all_data = {}

print("Starting data load...")

for year in years:
    # Construct the file path independent of the extension
    file_root = os.path.join(base_path, f"HH data {year}")
    
    try:
        if year in [2022, 2024]:
            # Load Stata files (.dta)
            file_path = f"{file_root}.dta"
            print(f"Loading {year} from .dta...")
            # convert_categoricals=False helps keep values as numbers (1,2) instead of labels ("Yes","No") initially
            df = pd.read_stata(file_path, convert_categoricals=False) 
            
        elif year == 2018:
            # Load Tab-separated .txt
            file_path = f"{file_root}.txt"
            print(f"Loading {year} from Tab-separated .txt...")
            df = pd.read_csv(file_path, sep='\t', encoding='latin1', low_memory=False)
            
        else:
            # Load Comma-separated .txt (2007-2016)
            file_path = f"{file_root}.txt"
            print(f"Loading {year} from Comma-separated .txt...")
            df = pd.read_csv(file_path, sep=',', encoding='latin1', low_memory=False)

        # Store in dictionary
        all_data[year] = df
        
    except FileNotFoundError:
        print(f"Could not find file for {year} at: {file_path}")

print("\n--- Load Complete ---")

# Verification: Print the shape of a few dataframes to ensure they loaded
if 2024 in all_data:
    print(f"df_2024 shape: {all_data[2024].shape}")
if 2007 in all_data:
    print(f"df_2007 shape: {all_data[2007].shape}")

Starting data load...
Loading 2007 from Comma-separated .txt...
Loading 2008 from Comma-separated .txt...
Loading 2009 from Comma-separated .txt...
Loading 2010 from Comma-separated .txt...
Loading 2011 from Comma-separated .txt...
Loading 2012 from Comma-separated .txt...
Loading 2013 from Comma-separated .txt...
Loading 2014 from Comma-separated .txt...
Loading 2016 from Comma-separated .txt...
Loading 2018 from Tab-separated .txt...
Loading 2022 from .dta...
Loading 2024 from .dta...

--- Load Complete ---
df_2024 shape: (646508, 74)
df_2007 shape: (735662, 65)


In [13]:
# Create a comparison of columns
# This creates a dictionary where keys are years and values are lists of columns
cols_dict = {year: pd.Series(df.columns) for year, df in all_data.items()}

# Convert to a DataFrame (rows will be uneven, that's fine)
cols_comparison = pd.concat(cols_dict, axis=1)

# Display the first few rows
print(cols_comparison.head())

# Optional: Save to Excel to look at it manually
cols_comparison.to_excel('ASER_Column_Mapping.xlsx')

            2007           2008           2009           2010           2011  \
0     state_code     state_code     state_code     state_code     state_code   
1     state_name  district_code  district_code     state_name     state_name   
2  district_code   village_code   village_code  district_code  district_code   
3  district_name     state_name     state_name  district_name  district_name   
4  aser05village  district_name  district_name   village_code   village_code   

            2012           2013           2014           2016           2018  \
0             ID             ID             ID     state_name     state_code   
1     state_code     state_name     state_name     state_code     state_name   
2     state_name     state_code     state_code  district_code  district_code   
3  district_code  district_code  district_code  district_name  district_name   
4  district_name  district_name  district_name   village_code   village_code   

            2022           2024  
0   

In [14]:
## 2024 Target Variables
target_vars = list(all_data[2024].columns)

In [26]:
#Defining functions

# Rename Variables
def rename_variables(df, rename_map):
    rename_map_existing = {k: v for k, v in rename_map.items() if k in df.columns}
    df = df.rename(columns=rename_map_existing)
    return df

# Convert Wide dummies to long Categorical Variables
def convert_wide_to_long(df, mapping):
    """
    mapping = {
        new_var: {dummy1: code1, dummy2: code2}
    }
    """
    for new_var, dummy_map in mapping.items():
        # Initialize target variable 
        df[new_var] = np.nan

        # For each dummy column in 2007
        for dummy_col, coded_val in dummy_map.items():
            if dummy_col in df.columns:
                df.loc[df[dummy_col] == 1, new_var] = coded_val

        # Drop original wide dummy columns
        cols_to_drop = [c for c in dummy_map if c in df.columns]
        if cols_to_drop:
            df.drop(columns=cols_to_drop, inplace=True)
    return df

# Add Missing Variables
def add_missing_vars(df, target_vars):
    for v in target_vars:
        if v not in df.columns:
            df[v] = np.nan
    return df

In [27]:
## 2007 ##

# Mapping Dictionary
rename_map_2007_to_2024 = {
    "village_id": "village_code",
    "totalmember": "total_member",
    "household_id": "hh_id",
    "child_slno": "child_no",
    "child_age": "child_age",
    "child_sex": "child_gender",
    "motherage": "mother_age",
    "motherstudieduptoclass": "mother_class",
    "languageinwhichchildtested": "test_language",
    "child5to16schoolingclassstd": "school_class"
    # No equivalent variables in 2024—they will be ignored:
    # aser05village, aser06village, aser07village, schtype_gender, mult
}

# Wide to Long
# school_type in 2024

wide_to_long_map_2007 = {
    # school_type in 2024

    "school_type": {
    "schoolstatusage5to16govt": 1,
    "schoolstatusage5to16pvt": 2,
    "schoolstatusage5to16madarsa": 3,
    "schoolstatusage5to16oth": 4,
    },
    "oos_never_enr": {
        "age5to16neverbeentoschool": 1
    },
    "oos_dropout": {
        "age5to16dropout": 1
    },
    "tuition": {
        "tutionyes": 1,
        "tutionno": 2
    },
    "surveyed_school": {
        "doeschildgotothesurveyedschoolye": 1,
        "doeschildgotothesurveyedschoolno": 2
    },
    "read_code": {
        "basiclearninglevelsreadnothing": 1,
        "basiclearninglevelsreadletter": 2,
        "basiclearninglevelsreadword": 3,
        "basiclearninglevelsreadpara": 4,   # para ≈ std1 level  
        "basiclearninglevelsreadstory": 5   # story ≈ std2 level  
    }
    ,
    "math_code": {
        "mathlevelnothing": 1,
        "mathlevelnumrec1to9": 2,
        "mathlevelnumrec10to99": 3,
        "mathlevelsubtraction": 4,
        "mathleveldivision": 5
    }
}


# === 2. Store in unified mapping object ===

mappings = {
    2007: {
        "rename_map": rename_map_2007_to_2024,
        "wide_to_long_map": wide_to_long_map_2007
    }
}

In [28]:
## 2008 ##

rename_map_2008_to_2024 = {

    # --- IDs & Geo ---
    "state_code": "state_code",
    "state_name": "state_name",
    "district_code": "district_code",
    "district_name": "district_name",
    "village_code": "village_code",
    "hh_id": "hh_id",
    "hh_no": "hh_no",
    "hh_multiplier": "hh_multiplier",
    "total_member": "total_member",

    # --- Household assets & type ---
    # 2024 has hh_type (1 pucca, 2 semi, 3 katcha)
    # 2008 has 3 separate dummy columns → will be handled in wide_to_long
    # No renaming needed here.

    "child_no": "child_no",
    "child_age": "child_age",
    "child_gender": "child_gender",

    # Mother schooling variables
    "mother_age": "mother_age",
    "mother_class": "mother_class",
    # mother_gone_to_school_yes/no → wide_to_long mapping

    # Pre-school
    # preschool_yes/no → maps to preschool_type (1=AWC, 2=Private nursery) — wide_to_long handles this

    "school_class": "school_class",
    # school_govt/private/madarsa/other → school_type via wide_to_long

    # Out-of-school
    "oos_never_enr": "oos_never_enr",
    "oos_dropout": "oos_dropout",
    "oos_dropout_class": "oos_dropout_class",
    "oos_dropout_year": "oos_dropout_year",

    # Learning levels
    # Will be handled by wide_to_long for:
    # read_code, math_code

    "test_language": "test_language",

    # Village infra
    "vlg_pucca_road": "vlg_pucca_road",
    "vlg_electricity_conn": "vlg_electricity",   # 2008 name → 2024 name
    "vlg_post_office": "vlg_post_office",
    "vlg_bank": "vlg_bank",
    "vlg_govt_primary_school": "vlg_govt_primary_school_1_5",
    "vlg_govt_middle_school": "vlg_govt_middle_school_1_8",
    "vlg_govt_secondary_school": "vlg_govt_secondary_school_1_10",
    "vlg_govt_private_school": "vlg_private_school"
}

wide_to_long_map_2008 = {

    # ----- Household type -----
    "hh_type": {
        "hhtype_pucca": 1,
        "hhtype_semi_katcha": 2,
        "hhtype_katcha": 3,
    },

    # ----- Mother schooling -----
    "mother_gone_to_school": {
        "mother_gone_to_school_yes": 1,
        "mother_gone_to_school_no": 2
    },

    # ----- Preschool type → 2024 scale (1 = anganwadi, 2 = private nursery) -----
    "preschool_type": {
        "preschool_yes": 1,   # assume govt/anganwadi
        "preschool_no": 2     # child not attending → coded as “other”
    },

    # ----- School type (Govt, Pvt, Madarsa, Other) -----
    "school_type": {
        "school_govt": 1,
        "school_private": 2,
        "school_madarsa": 3,
        "school_other": 4
    },

    # ----- Reading -----
    "read_code": {
        "read_nothing": 1,
        "read_letter": 2,
        "read_word": 3,
        "read_level_1": 4,   # Std 1 text
        "read_level_2": 5    # Std 2 text
    },

    # ----- Math -----
    "math_code": {
        "math_nothing": 1,
        "math_num_1_9": 2,
        "math_num_10_99": 3,
        "math_subtraction": 4,
        "math_division": 5
    }
}

mappings[2008] = {
    "rename_map": rename_map_2008_to_2024,
    "wide_to_long_map": wide_to_long_map_2008
}

In [29]:
## 2009 CORRECTED ##

rename_map_2009_to_2024 = {
    # --- Identifiers ---
    "state_name": "state_name",
    "state_code": "state_code",
    "district_code": "district_code",
    "district_name": "district_name",
    "village_code": "village_code",
    "hh_id": "hh_id",
    "hh_no": "hh_no",
    "hh_multiplier": "hh_multiplier",
    "total_member": "total_member",

    # --- Household Assets ---
    # NOTE: 2009 uses counts (0,1,2...), 2024 uses Yes/No (1/2). 
    # Renaming aligns columns, but you must manually threshold values later.
    "hh_motor_cycle": "hh_motor_veh_2whl",  
    "hh_four_wheeler": "hh_motor_veh_4whl",
    
    # 2009 specific vars mapped to 2024 names directly
    "hh_electricity_today": "hh_electricity_today",
    "hh_toilet": "hh_toilet",
    
    # --- Child Identifiers ---
    "child_no": "child_no",
    "mother_id": "mother_no",
    "child_age": "child_age",
    "child_gender": "child_gender",

    # --- Schooling ---
    "school_class": "school_class",
    "tuition": "tuition",
    "surveyed_school": "surveyed_school",
    "test_language": "test_language",

    # --- Out of School (OOS) ---
    "oos_never_enr": "oos_never_enr",
    "oos_dropout": "oos_dropout",
    "oos_dropout_class": "oos_dropout_class",
    "oos_dropout_year": "oos_dropout_year",

    # --- Parents ---
    "father_age": "father_age",
    "father_gone_to_school": "father_gone_to_school",
    "father_class": "father_class",
    "mother_age": "mother_age",
    "mother_gone_to_school": "mother_gone_to_school",
    "mother_class": "mother_class",

    # --- Village Infrastructure ---
    "vlg_pucca_road": "vlg_pucca_road",
    "vlg_electricity": "vlg_electricity",
    "vlg_post_office": "vlg_post_office",
    "vlg_bank": "vlg_bank",
    "vlg_govt_primary_health_clinic": "vlg_govt_primary_health_clinic",
    "vlg_private_health_clinic": "vlg_private_health_clinic",
    
    # Mapping specific school levels to 2024 names
    "vlg_govt_primary_school": "vlg_govt_primary_school_1_5",
    "vlg_govt_middle_school": "vlg_govt_middle_school_1_8",
    "vlg_govt_secondary_school": "vlg_govt_secondary_school_1_10",
    "vlg_private_school": "vlg_private_school",
    "vlg_anganwadi": "vlg_anganwadi",
}

wide_to_long_map_2009 = {
    # Format: { TARGET_VAR: { OLD_DUMMY: NEW_CODE } }

    # --- Household Type ---
    # Target: 1=Pucca, 2=Semi-Pucca, 3=Katcha
    "hh_type": {
        "hhtype_pucca": 1,
        "hhtype_semi_katcha": 2,
        "hhtype_katcha": 3
    },

    # --- Electricity Connection ---
    # Target: 1=Yes, 2=No
    "hh_electricity_conn": {
        "hh_electricity_conn_yes": 1,
        "hh_electricity_conn_no": 2
    },

    # --- TV ---
    # Target: 1=Yes, 2=No
    "hh_tv": {
        "hh_tv_yes": 1,
        "hh_tv_no": 2
    },

    # --- Mobile ---
    # Target: 1=Yes, 2=No
    "hh_mobile": {
        "hh_mobile_yes": 1,
        "hh_mobile_no": 2
    },

    # --- Preschool Type ---
    # Target: 1=Anganwadi (Govt), 2=Kindergarten (LKG/UKG)
    "preschool_type": {
        "preschool_yes": 1,       # "Child is going to pre-school" usually implies Anganwadi in this context
        "kindergarton_yes": 2     # "Child going LKG/UKG"
    },

    # --- School Type ---
    # Target: 1=Govt, 2=Pvt, 3=Madarsa, 4=Other/EGS
    "school_type": {
        "school_govt": 1,
        "school_private": 2,
        "school_madarsa": 3,
        "school_other": 4
    },

    # --- Reading Levels ---
    # Target: 1=Nothing, 2=Letter, 3=Word, 4=Std1, 5=Std2
    "read_code": {
        "read_nothing": 1,
        "read_letter": 2,
        "read_word": 3,
        "read_level_1": 4,
        "read_level_2": 5
    },

    # --- Math Levels ---
    # Target: 1=Nothing, 2=1-9, 3=10-99, 4=Sub, 5=Div
    "math_code": {
        "math_nothing": 1,
        "math_num_1_9": 2,
        "math_num_10_99": 3,
        "math_subtraction": 4,
        "math_division": 5
    }
}

mappings[2009] = {
    "rename_map": rename_map_2009_to_2024,
    "wide_to_long_map": wide_to_long_map_2009
}

In [30]:
## 2010 ##

rename_map_2010_to_2024 = {
    # --- Identifiers ---
    "state_name": "state_name",
    "state_code": "state_code",
    "district_code": "district_code",
    "district_name": "district_name",
    "village_code": "village_code",
    "hh_id": "hh_id",
    "hh_no": "hh_no",
    "hh_multiplier": "hh_multiplier",
    "total_member": "total_member",

    # --- Child Identifiers ---
    "child_no": "child_no",
    "mother_no": "mother_no",
    "child_age": "child_age",
    "child_gender": "child_gender",

    # --- Schooling (General) ---
    "school_class": "school_class",
    "test_language": "test_language",

    # --- Out of School ---
    "oos_never_enr": "oos_never_enr",
    "oos_dropout": "oos_dropout",
    "oos_dropout_class": "oos_dropout_class",
    "oos_dropout_year": "oos_dropout_year",

    # --- Parents (Demographics only here, education is in wide_to_long) ---
    "father_age": "father_age",
    "father_class": "father_class",
    "mother_age": "mother_age",
    "mother_class": "mother_class",

    # --- Village Infrastructure ---
    "vlg_pucca_road": "vlg_pucca_road",
    "vlg_electricity": "vlg_electricity",
    "vlg_post_office": "vlg_post_office",
    "vlg_bank": "vlg_bank",
    "vlg_govt_primary_health_clinic": "vlg_govt_primary_health_clinic",
    "vlg_private_health_clinic": "vlg_private_health_clinic",
    "vlg_internet_cafe": "vlg_internet_cafe",
    
    # Mapping specific school levels to 2024 names
    "vlg_govt_primary_school": "vlg_govt_primary_school_1_5",
    "vlg_govt_middle_school": "vlg_govt_middle_school_1_8",
    "vlg_govt_secondary_school": "vlg_govt_secondary_school_1_10",
    "vlg_private_school": "vlg_private_school",
    "vlg_anganwadi": "vlg_anganwadi",
}

wide_to_long_map_2010 = {
    # --- Household Type ---
    # Target: 1=Pucca, 2=Semi-Pucca, 3=Katcha
    "hh_type": {
        "hhtype_pucca": 1,
        "hhtype_semi_katcha": 2,
        "hhtype_katcha": 3
    },

    # --- Household Assets (Consolidating Yes/No pairs) ---
    # Target: 1=Yes, 2=No
    "hh_electricity_conn": {
        "hh_electricity_conn_yes": 1,
        "hh_electricity_conn_no": 2
    },
    "hh_electricity_today": {
        "hh_electricity_today_yes": 1,
        "hh_electricity_today_no": 2
    },
    "hh_toilet": {
        "hh_toilet_yes": 1,
        "hh_toilet_no": 2
    },
    "hh_tv": {
        "hh_tv_yes": 1,
        "hh_tv_no": 2
    },
    "hh_mobile": {
        "hh_mobile_yes": 1,
        "hh_mobile_no": 2
    },
    "hh_newspaper": {
        "hh_newspaper_yes": 1,
        "hh_newspaper_no": 2
    },
    "hh_reading_material": {
        "hh_reading_material_yes": 1,
        "hh_reading_material_no": 2
    },
    "hh_computer_use": {
        "hh_computer_use_yes": 1,
        "hh_computer_use_no": 2
    },

    # --- Preschool Type ---
    # Target: 1=Anganwadi, 2=Kindergarten
    "preschool_type": {
        "preschool_yes": 1,
        "kindergarton_yes": 2
    },

    # --- School Type ---
    # Target: 1=Govt, 2=Pvt, 3=Madarsa, 4=Other
    "school_type": {
        "school_govt": 1,
        "school_private": 2,
        "school_madarsa": 3,
        "school_other": 4
    },

    # --- Tuition ---
    # Target: 1=Yes, 2=No
    "tuition": {
        "tuition_yes": 1,
        "tuition_no": 2
    },

    # --- Surveyed School ---
    # Target: 1=Yes, 2=No
    "surveyed_school": {
        "surveyed_school_yes": 1,
        "surveyed_school_no": 2
    },

    # --- Parents Education (Binary) ---
    # Target: 1=Yes, 2=No
    "father_gone_to_school": {
        "father_gone_to_school_yes": 1,
        "father_gone_to_school_no": 2
    },
    "mother_gone_to_school": {
        "mother_gone_to_school_yes": 1,
        "mother_gone_to_school_no": 2
    },

    # --- Reading Levels ---
    # Target: 1=Nothing ... 5=Std2
    "read_code": {
        "read_nothing": 1,
        "read_letter": 2,
        "read_word": 3,
        "read_level_1": 4,
        "read_level_2": 5
    },

    # --- Math Levels ---
    # Target: 1=Nothing ... 5=Division
    "math_code": {
        "math_nothing": 1,
        "math_num_1_9": 2,
        "math_num_10_99": 3,
        "math_subtraction": 4,
        "math_division": 5
    }
}

mappings[2010] = {
    "rename_map": rename_map_2010_to_2024,
    "wide_to_long_map": wide_to_long_map_2010
}

In [31]:
## 2011 ##

rename_map_2011_to_2024 = {
    # --- Identifiers ---
    "state_name": "state_name",
    "state_code": "state_code",
    "district_code": "district_code",
    "district_name": "district_name",
    "village_code": "village_code",
    "hh_id": "hh_id",
    "hh_no": "hh_no",
    "hh_multiplier": "hh_multiplier",
    "total_member": "total_member",
    
    # --- Assets (Already 1/2 Yes/No) ---
    # Note: hh_type values are inverted (1=Katcha) vs 2024 (1=Pucca)
    "hh_type": "hh_type", 
    "hh_electricity_conn": "hh_electricity_conn",
    "hh_electricity_today": "hh_electricity_today",
    "hh_toilet": "hh_toilet",
    "hh_tv": "hh_tv",
    "hh_mobile": "hh_mobile",
    "hh_newspaper": "hh_newspaper",
    "hh_reading_material": "hh_reading_material",
    "hh_computer_use": "hh_computer_use",

    # --- Child Identifiers ---
    "child_no": "child_no",
    "mother_no": "mother_no",
    "child_age": "child_age",
    "child_gender": "child_gender",

    # --- Schooling ---
    "school_class": "school_class",
    "tuition": "tuition",
    "surveyed_school": "surveyed_school",
    "test_language": "test_language",

    # --- Out of School ---
    "oos_never_enr": "oos_never_enr",
    "oos_dropout": "oos_dropout",
    "oos_dropout_class": "oos_dropout_class",
    "oos_dropout_year": "oos_dropout_year",

    # --- Parents (Demographics) ---
    "father_age": "father_age",
    "father_class": "father_class",
    "mother_age": "mother_age",
    "mother_class": "mother_class",

    # --- Village ---
    "vlg_pucca_road": "vlg_pucca_road",
    "vlg_electricity": "vlg_electricity",
    "vlg_post_office": "vlg_post_office",
    "vlg_bank": "vlg_bank",
    "vlg_govt_primary_health_clinic": "vlg_govt_primary_health_clinic",
    "vlg_private_health_clinic": "vlg_private_health_clinic",
    "vlg_internet_cafe": "vlg_internet_cafe",
    "vlg_govt_primary_school": "vlg_govt_primary_school_1_5",
    "vlg_govt_middle_school": "vlg_govt_middle_school_1_8",
    "vlg_govt_secondary_school": "vlg_govt_secondary_school_1_10",
    "vlg_private_school": "vlg_private_school",
    "vlg_anganwadi": "vlg_anganwadi",
}

wide_to_long_map_2011 = {
    # --- Preschool ---
    "preschool_type": {
        "preschool_yes": 1,
        "kindergarton_yes": 2
    },

    # --- School Type ---
    "school_type": {
        "school_govt": 1,
        "school_private": 2,
        "school_madarsa": 3,
        "school_other": 4
    },

    # --- Reading ---
    "read_code": {
        "read_nothing": 1,
        "read_letter": 2,
        "read_word": 3,
        "read_level_1": 4,
        "read_level_2": 5
    },

    # --- Math ---
    "math_code": {
        "math_nothing": 1,
        "math_num_1_9": 2,
        "math_num_10_99": 3,
        "math_subtraction": 4,
        "math_division": 5
    },

    # --- Parents Education (Binary dummies -> 1/2) ---
    "father_gone_to_school": {
        "father_gone_to_school_yes": 1,
        "father_gone_to_school_no": 2
    },
    "mother_gone_to_school": {
        "mother_gone_to_school_yes": 1,
        "mother_gone_to_school_no": 2
    }
}

mappings[2011] = {
    "rename_map": rename_map_2011_to_2024,
    "wide_to_long_map": wide_to_long_map_2011
}

In [32]:
## 2012 ##

rename_map_2012_to_2024 = {
    "state_name": "state_name",
    "state_code": "state_code",
    "district_code": "district_code",
    "district_name": "district_name",
    "village_code": "village_code",
    "hh_id": "hh_id",
    "hh_no": "hh_no",
    "hh_multiplier": "hh_multiplier",
    "total_member": "total_member",

    # Assets
    "hh_type": "hh_type", # Value inverted (1=Katcha)
    "hh_electricity_conn": "hh_electricity_conn",
    "hh_electricity_today": "hh_electricity_today",
    "hh_toilet": "hh_toilet",
    "hh_tv": "hh_tv",
    "hh_mobile": "hh_mobile",
    "hh_newspaper": "hh_newspaper",
    "hh_reading_material": "hh_reading_material",
    "hh_computer_use": "hh_computer_use",
    "hh_motor_vehicle": "hh_motor_veh_2whl", # Specific to 2-wheeler in description

    # Child
    "child_no": "child_no",
    "mother_no": "mother_no",
    "child_age": "child_age",
    "child_gender": "child_gender",

    # Schooling
    "school_class": "school_class",
    "tuition": "tuition",
    "surveyed_school": "surveyed_school",
    "test_language": "test_language",

    # OOS
    "oos_never_enr": "oos_never_enr",
    "oos_dropout": "oos_dropout",
    "oos_dropout_class": "oos_dropout_class",
    "oos_dropout_year": "oos_dropout_year",

    # Parents
    "father_age": "father_age",
    "father_class": "father_class",
    "mother_age": "mother_age",
    "mother_class": "mother_class",

    # Village
    "vlg_pucca_road": "vlg_pucca_road",
    "vlg_electricity": "vlg_electricity",
    "vlg_post_office": "vlg_post_office",
    "vlg_bank": "vlg_bank",
    "vlg_govt_primary_health_clinic": "vlg_govt_primary_health_clinic",
    "vlg_private_health_clinic": "vlg_private_health_clinic",
    "vlg_internet_cafe": "vlg_internet_cafe",
    "vlg_govt_primary_school_1_5": "vlg_govt_primary_school_1_5",
    "vlg_govt_middle_school_1_8": "vlg_govt_middle_school_1_8",
    "vlg_govt_secondary_school_1_10": "vlg_govt_secondary_school_1_10",
    "vlg_private_school": "vlg_private_school",
    "vlg_anganwadi": "vlg_anganwadi",
}

wide_to_long_map_2012 = {
    "preschool_type": {
        "preschool_yes": 1,
        "kindergarton_yes": 2
    },
    "school_type": {
        "school_govt": 1,
        "school_private": 2,
        "school_madarsa": 3,
        "school_other": 4
    },
    "read_code": {
        "read_nothing": 1,
        "read_letter": 2,
        "read_word": 3,
        "read_level_1": 4,
        "read_level_2": 5
    },
    "math_code": {
        "math_nothing": 1,
        "math_num_1_9": 2,
        "math_num_10_99": 3,
        "math_subtraction": 4,
        "math_division": 5
    },
    # Note: 2024 does not usually list English, but we map it here for completeness
    "english_code": {
        "english_nothing": 1,
        "english_uppercase_letter": 2,
        "english_lowercase_letter": 3,
        "english_word": 4,
        "english_sentence": 5
    },
    "father_gone_to_school": {
        "father_gone_to_school_yes": 1,
        "father_gone_to_school_no": 2
    },
    "mother_gone_to_school": {
        "mother_gone_to_school_yes": 1,
        "mother_gone_to_school_no": 2
    }
}

mappings[2012] = {
    "rename_map": rename_map_2012_to_2024,
    "wide_to_long_map": wide_to_long_map_2012
}

In [33]:
## 2014 ##

rename_map_2014_to_2024 = {
    "state_name": "state_name",
    "state_code": "state_code",
    "district_code": "district_code",
    "district_name": "district_name",
    "village_code": "village_code",
    "hh_id": "hh_id",
    "hh_no": "hh_no",
    "hh_multiplier": "hh_multiplier",
    "total_member": "total_member",

    # Assets
    "hh_type": "hh_type", # Value inverted
    "hh_electricity_conn": "hh_electricity_conn",
    "hh_electricity_today": "hh_electricity_today",
    "hh_toilet": "hh_toilet",
    "hh_tv": "hh_tv",
    "hh_mobile": "hh_mobile",
    "hh_newspaper": "hh_newspaper",
    "hh_reading_material": "hh_reading_material",
    "hh_grad": "hh_grad",
    "hh_computer_use": "hh_computer_use",
    "hh_motor_vehicle": "hh_motor_veh_2whl",

    # Child
    "child_no": "child_no",
    "mother_no": "mother_no",
    "child_age": "child_age",
    "child_gender": "child_gender",

    # Schooling
    "school_class": "school_class",
    "tuition": "tuition",
    "surveyed_school": "surveyed_school",
    "test_language": "test_language",

    # Learning (Categorical)
    "read_code": "read_code",
    "math_code": "math_code",
    "english_code": "english_code", 

    # OOS
    "oos_never_enr": "oos_never_enr",
    "oos_dropout": "oos_dropout",
    "oos_dropout_class": "oos_dropout_class",
    "oos_dropout_year": "oos_dropout_year",

    # Parents (NOW CATEGORICAL)
    "father_age": "father_age",
    "father_gone_to_school": "father_gone_to_school",
    "father_class": "father_class",
    "mother_age": "mother_age",
    "mother_gone_to_school": "mother_gone_to_school",
    "mother_class": "mother_class",

    # Village
    "vlg_pucca_road": "vlg_pucca_road",
    "vlg_electricity": "vlg_electricity",
    "vlg_post_office": "vlg_post_office",
    "vlg_bank": "vlg_bank",
    "vlg_govt_primary_health_clinic": "vlg_govt_primary_health_clinic",
    "vlg_private_health_clinic": "vlg_private_health_clinic",
    "vlg_internet_cafe": "vlg_internet_cafe",
    "vlg_govt_primary_school_1_5": "vlg_govt_primary_school_1_5",
    "vlg_govt_middle_school_1_8": "vlg_govt_middle_school_1_8",
    "vlg_govt_secondary_school_1_10": "vlg_govt_secondary_school_1_10",
    "vlg_private_school": "vlg_private_school",
    "vlg_anganwadi": "vlg_anganwadi",
}

wide_to_long_map_2014 = {
    "preschool_type": {
        "preschool_yes": 1,
        "kindergarton_yes": 2
    },
    "school_type": {
        "school_govt": 1,
        "school_private": 2,
        "school_madarsa": 3,
        "school_other": 4
    }
}

mappings[2014] = {
    "rename_map": rename_map_2014_to_2024,
    "wide_to_long_map": wide_to_long_map_2014
}

In [34]:
## 2013 ##

rename_map_2013_to_2024 = {
    "state_name": "state_name",
    "state_code": "state_code",
    "district_code": "district_code",
    "district_name": "district_name",
    "village_code": "village_code",
    "hh_id": "hh_id",
    "hh_no": "hh_no",
    "hh_multiplier": "hh_multiplier",
    "total_member": "total_member",

    # Assets
    "hh_type": "hh_type", # Value inverted
    "hh_electricity_conn": "hh_electricity_conn",
    "hh_electricity_today": "hh_electricity_today",
    "hh_toilet": "hh_toilet",
    "hh_tv": "hh_tv",
    "hh_mobile": "hh_mobile",
    "hh_newspaper": "hh_newspaper",
    "hh_reading_material": "hh_reading_material",
    "hh_grad": "hh_grad",
    "hh_computer_use": "hh_computer_use",
    "hh_motor_vehicle": "hh_motor_veh_2whl",

    # Child
    "child_no": "child_no",
    "mother_no": "mother_no",
    "child_age": "child_age",
    "child_gender": "child_gender",

    # Schooling
    "school_class": "school_class",
    "tuition": "tuition",
    "surveyed_school": "surveyed_school",
    "test_language": "test_language",

    # Learning (NOW CATEGORICAL 1-5)
    "read_code": "read_code",
    "math_code": "math_code",

    # OOS
    "oos_never_enr": "oos_never_enr",
    "oos_dropout": "oos_dropout",
    "oos_dropout_class": "oos_dropout_class",
    "oos_dropout_year": "oos_dropout_year",

    # Parents
    "father_age": "father_age",
    "father_class": "father_class",
    "mother_age": "mother_age",
    "mother_class": "mother_class",

    # Village
    "vlg_pucca_road": "vlg_pucca_road",
    "vlg_electricity": "vlg_electricity",
    "vlg_post_office": "vlg_post_office",
    "vlg_bank": "vlg_bank",
    "vlg_govt_primary_health_clinic": "vlg_govt_primary_health_clinic",
    "vlg_private_health_clinic": "vlg_private_health_clinic",
    "vlg_internet_cafe": "vlg_internet_cafe",
    "vlg_govt_primary_school_1_5": "vlg_govt_primary_school_1_5",
    "vlg_govt_middle_school_1_8": "vlg_govt_middle_school_1_8",
    "vlg_govt_secondary_school_1_10": "vlg_govt_secondary_school_1_10",
    "vlg_private_school": "vlg_private_school",
    "vlg_anganwadi": "vlg_anganwadi",
}

wide_to_long_map_2013 = {
    "preschool_type": {
        "preschool_yes": 1,
        "kindergarton_yes": 2
    },
    "school_type": {
        "school_govt": 1,
        "school_private": 2,
        "school_madarsa": 3,
        "school_other": 4
    },
    "father_gone_to_school": {
        "father_gone_to_school_yes": 1,
        "father_gone_to_school_no": 2
    },
    "mother_gone_to_school": {
        "mother_gone_to_school_yes": 1,
        "mother_gone_to_school_no": 2
    }
}

mappings[2013] = {
    "rename_map": rename_map_2013_to_2024,
    "wide_to_long_map": wide_to_long_map_2013
}

In [35]:
## 2016 ##

rename_map_2016_to_2024 = {
    # Identifiers
    "state_name": "state_name",
    "state_code": "state_code",
    "district_code": "district_code",
    "district_name": "district_name",
    "village_code": "village_code",
    "hh_id": "hh_id",
    "hh_no": "hh_no",
    "hh_multiplier": "hh_multiplier",
    "total_member": "total_member",

    # Assets (1=Yes, 2=No)
    "hh_type": "hh_type", # NOTE: Inverted (1=Katcha)
    "hh_electricity_conn": "hh_electricity_conn",
    "hh_electricity_today": "hh_electricity_today",
    "hh_toilet": "hh_toilet",
    "hh_tv": "hh_tv",
    "hh_mobile": "hh_mobile",
    "hh_newspaper": "hh_newspaper",
    "hh_reading_material": "hh_reading_material",
    "hh_grad": "hh_grad",
    "hh_computer_use": "hh_computer_use",
    "hh_motor_veh_4whl": "hh_motor_veh_4whl",
    "hh_motor_veh_2whl": "hh_motor_veh_2whl",

    # Child
    "child_no": "child_no",
    "mother_no": "mother_no",
    "child_age": "child_age",
    "child_gender": "child_gender",

    # Schooling
    "school_class": "school_class",
    "tuition": "tuition",
    "surveyed_school": "surveyed_school",
    "test_language": "test_language",

    # Learning (Categorical 1-5)
    "read_code": "read_code",
    "math_code": "math_code",
    "english_code": "english_code",

    # OOS
    "oos_never_enr": "oos_never_enr",
    "oos_dropout": "oos_dropout",
    "oos_dropout_class": "oos_dropout_class",
    "oos_dropout_year": "oos_dropout_year",

    # Parents
    "father_age": "father_age",
    "father_gone_to_school": "father_gone_to_school",
    "father_class": "father_class",
    "mother_age": "mother_age",
    "mother_gone_to_school": "mother_gone_to_school",
    "mother_class": "mother_class",

    # Village
    "vlg_pucca_road": "vlg_pucca_road",
    "vlg_electricity": "vlg_electricity",
    "vlg_post_office": "vlg_post_office",
    "vlg_bank": "vlg_bank",
    "vlg_govt_primary_health_clinic": "vlg_govt_primary_health_clinic",
    "vlg_private_health_clinic": "vlg_private_health_clinic",
    "vlg_internet_cafe": "vlg_internet_cafe",
    "vlg_govt_primary_school_1_5": "vlg_govt_primary_school_1_5",
    "vlg_govt_middle_school_1_8": "vlg_govt_middle_school_1_8",
    "vlg_govt_secondary_school_1_10": "vlg_govt_secondary_school_1_10",
    "vlg_private_school": "vlg_private_school",
    "vlg_anganwadi": "vlg_anganwadi",
}

wide_to_long_map_2016 = {
    # Preschool (Dummy -> Categorical)
    "preschool_type": {
        "preschool_yes": 1,
        "kindergarton_yes": 2
    },
    # School Type (Dummy -> Categorical)
    "school_type": {
        "school_govt": 1,
        "school_private": 2,
        "school_madarsa": 3,
        "school_other": 4
    }
}

mappings[2016] = {
    "rename_map": rename_map_2016_to_2024,
    "wide_to_long_map": wide_to_long_map_2016
}

In [36]:
## 2018 ##

rename_map_2018_to_2024 = {
    "state_name": "state_name",
    "state_code": "state_code",
    "district_code": "district_code",
    "district_name": "district_name",
    "village_code": "village_code",
    "hh_id": "hh_id",
    "hh_no": "hh_no",
    "hh_multiplier": "hh_multiplier",
    "total_member": "total_member",

    # Assets
    "hh_type": "hh_type", # NOTE: Inverted (1=Katcha)
    "hh_electricity_conn": "hh_electricity_conn",
    "hh_electricity_today": "hh_electricity_today",
    "hh_toilet": "hh_toilet",
    "hh_tv": "hh_tv",
    "hh_mobile": "hh_mobile",
    "hh_mobile_smart": "hh_mobile_smart", # New in 2018
    "hh_newspaper": "hh_newspaper",
    "hh_reading_material": "hh_reading_material",
    "hh_grad": "hh_grad",
    "hh_computer_use": "hh_computer_use",
    "hh_motor_veh_4whl": "hh_motor_veh_4whl",
    "hh_motor_veh_2whl": "hh_motor_veh_2whl",

    # Child
    "child_no": "child_no",
    "mother_no": "mother_no",
    "child_age": "child_age",
    "child_gender": "child_gender",

    # Schooling (Direct Renames now)
    "preschool_type": "preschool_type",
    "school_type": "school_type",
    "school_class": "school_class",
    "tuition": "tuition",
    "surveyed_school": "surveyed_school",
    "test_language": "test_language",
    "school_medium_instruction": "school_medium_instruction",

    # Learning
    "read_code": "read_code",
    "math_code": "math_code",

    # OOS
    "oos_never_enr": "oos_never_enr",
    "oos_dropout": "oos_dropout",
    "oos_dropout_class": "oos_dropout_class",
    "oos_dropout_year": "oos_dropout_year",

    # Parents
    "father_age": "father_age",
    "father_gone_to_school": "father_gone_to_school",
    "father_class": "father_class",
    "mother_age": "mother_age",
    "mother_gone_to_school": "mother_gone_to_school",
    "mother_class": "mother_class",

    # Village
    "vlg_pucca_road": "vlg_pucca_road",
    "vlg_electricity": "vlg_electricity",
    "vlg_post_office": "vlg_post_office",
    "vlg_bank": "vlg_bank",
    "vlg_govt_primary_health_clinic": "vlg_govt_primary_health_clinic",
    "vlg_private_health_clinic": "vlg_private_health_clinic",
    "vlg_internet_cafe": "vlg_internet_cafe",
    "vlg_govt_primary_school_1_5": "vlg_govt_primary_school_1_5",
    "vlg_govt_middle_school_1_8": "vlg_govt_middle_school_1_8",
    "vlg_govt_secondary_school_1_10": "vlg_govt_secondary_school_1_10",
    "vlg_private_school": "vlg_private_school",
    "vlg_anganwadi": "vlg_anganwadi",
}

# No wide_to_long needed for 2018
mappings[2018] = {
    "rename_map": rename_map_2018_to_2024,
    "wide_to_long_map": {} 
}

In [37]:
## 2022 ##

rename_map_2022_to_2024 = {
    "state_name": "state_name",
    "state_code": "state_code",
    "district_code": "district_code",
    "district_name": "district_name",
    "village_code": "village_code",
    "hh_id": "hh_id",
    "hh_no": "hh_no",
    "hh_multiplier": "hh_multiplier",
    "total_member": "total_member",

    # Assets
    "hh_type": "hh_type", # NOTE: MATCHES 2024 (1=Pucca). No Swap needed.
    "hh_electricity_conn": "hh_electricity_conn",
    "hh_electricity_today": "hh_electricity_today",
    "hh_toilet": "hh_toilet",
    "hh_tv": "hh_tv",
    "hh_mobile": "hh_mobile",
    "hh_mobile_smart": "hh_mobile_smart",
    "hh_mobile_smart_num": "hh_mobile_smart_num",
    "hh_internet_today": "hh_internet_today",
    "hh_newspaper": "hh_newspaper",
    "hh_reading_material": "hh_reading_material",
    "hh_grad": "hh_grad",
    "hh_computer_use": "hh_computer_use",
    "hh_motor_veh_4whl": "hh_motor_veh_4whl",
    "hh_motor_veh_2whl": "hh_motor_veh_2whl",

    # Child
    "child_no": "child_no",
    "mother_no": "mother_no",
    "child_age": "child_age",
    "child_gender": "child_gender",

    # Schooling
    "preschool_type": "preschool_type",
    "school_type": "school_type",
    "school_class": "school_class",
    "tuition": "tuition",
    "surveyed_school": "surveyed_school",
    "test_language": "test_language",
    "school_medium_instruction": "school_medium_instruction",

    # Learning
    "read_code": "read_code",
    "math_code": "math_code",
    "english_code": "english_code",

    # OOS
    "oos_never_enr": "oos_never_enr",
    "oos_dropout": "oos_dropout",
    "oos_dropout_class": "oos_dropout_class",
    "oos_dropout_year": "oos_dropout_year",

    # Parents
    "father_age": "father_age",
    "father_gone_to_school": "father_gone_to_school",
    "father_class": "father_class",
    "mother_age": "mother_age",
    "mother_gone_to_school": "mother_gone_to_school",
    "mother_class": "mother_class",

    # Village
    "vlg_pucca_road": "vlg_pucca_road",
    "vlg_electricity": "vlg_electricity",
    "vlg_post_office": "vlg_post_office",
    "vlg_bank": "vlg_bank",
    "vlg_govt_primary_health_clinic": "vlg_govt_primary_health_clinic",
    "vlg_private_health_clinic": "vlg_private_health_clinic",
    "vlg_internet_cafe": "vlg_internet_cafe",
    "vlg_govt_primary_school_1_5": "vlg_govt_primary_school_1_5",
    "vlg_govt_middle_school_1_8": "vlg_govt_middle_school_1_8",
    "vlg_govt_secondary_school_1_10": "vlg_govt_secondary_school_1_10",
    "vlg_private_school": "vlg_private_school",
    "vlg_anganwadi": "vlg_anganwadi",
}

mappings[2022] = {
    "rename_map": rename_map_2022_to_2024,
    "wide_to_long_map": {}
}

In [42]:
## 2024 ##
# Since 2024 is the target standard, we pass empty maps.
# This ensures it passes the loop check but remains unchanged.

mappings[2024] = {
    "rename_map": {},       # No renaming needed
    "wide_to_long_map": {}  # No conversion needed
}

In [43]:
# --- PRE-PROCESSING FIXES ---

# 1. Fix hh_type (Your code)
years_to_fix_type = [2011, 2012, 2013, 2014, 2016, 2018]
type_swap_map = {1: 3, 3: 1} # 2 stays 2

for year in years_to_fix_type:
    if year in all_data and 'hh_type' in all_data[year].columns:
        print(f"Fixing hh_type for {year}...")
        all_data[year]['hh_type'] = all_data[year]['hh_type'].replace(type_swap_map)

# 2. Fix 2009 Vehicle Counts -> Binaries (1=Yes, 2=No)
if 2009 in all_data:
    print("Fixing 2009 vehicle counts...")
    df_09 = all_data[2009]
    
    # Logic: If count > 0 then 1 (Yes), else 2 (No)
    # We must do this BEFORE renaming, using the OLD names
    if 'hh_motor_cycle' in df_09.columns:
        df_09['hh_motor_cycle'] = np.where(df_09['hh_motor_cycle'] > 0, 1, 2)
        
    if 'hh_four_wheeler' in df_09.columns:
        df_09['hh_four_wheeler'] = np.where(df_09['hh_four_wheeler'] > 0, 1, 2)
        
    all_data[2009] = df_09

Fixing hh_type for 2011...
Fixing hh_type for 2012...
Fixing hh_type for 2013...
Fixing hh_type for 2014...
Fixing hh_type for 2016...
Fixing hh_type for 2018...
Fixing 2009 vehicle counts...


In [44]:
new_data = {}

for year, df in all_data.items():
    
    if year not in mappings:
        print(f"Skipping {year} — no mapping defined yet")
        continue
    
    print(f"Processing {year}...")
    
    df = rename_variables(df, mappings[year]["rename_map"])
    df = convert_wide_to_long(df, mappings[year]["wide_to_long_map"])
    df = add_missing_vars(df, target_vars)
    
    new_data[year] = df[target_vars]

print("\n--- Processing Complete ---")

Processing 2007...
Processing 2008...
Processing 2009...
Processing 2010...
Processing 2011...
Processing 2012...
Processing 2013...
Processing 2014...
Processing 2016...
Processing 2018...
Processing 2022...
Processing 2024...

--- Processing Complete ---


In [45]:
# Create a list to hold the processed dataframes
dfs_to_merge = []

for year, df in new_data.items():
    # Important: Create a 'year' column so you know which dataset the row came from
    # We do this using .assign() to avoid SettingWithCopy warnings
    df_with_year = df.assign(year=year)
    dfs_to_merge.append(df_with_year)

# Concatenate all dataframes into one master dataframe
# ignore_index=True resets the index to 0, 1, 2... instead of keeping old row numbers
master_data = pd.concat(dfs_to_merge, ignore_index=True)

# Sort by year and state/district if desired (optional)
# Using generic column names since I know you have state_name and district_name
sort_cols = [c for c in ['year', 'state_name', 'district_name'] if c in master_data.columns]
if sort_cols:
    master_data = master_data.sort_values(by=sort_cols)

print("--- Merge Complete ---")
print(f"Total shape: {master_data.shape}")
print(f"Years included: {master_data['year'].unique()}")

# Optional: Preview the data
print(master_data.head())

# Optional: Export to Stata or CSV
# master_data.to_stata("ASER_Panel_2007_2024.dta", version=118)
# master_data.to_csv("ASER_Panel_2007_2024.csv", index=False)

--- Merge Complete ---
Total shape: (8016562, 75)
Years included: [2007 2008 2009 2010 2011 2012 2013 2014 2016 2018 2022 2024]
                 state_name  state_code  district_code  \
24463  ARUNACHAL PRADESH           12.0           12.0   
24464  ARUNACHAL PRADESH           12.0           12.0   
24465  ARUNACHAL PRADESH           12.0           12.0   
24466  ARUNACHAL PRADESH           12.0           12.0   
24467  ARUNACHAL PRADESH           12.0           12.0   

                      district_name  village_code  aser24  hh_no     hh_id  \
24463  Changlang                            690.0     NaN    NaN  690001.0   
24464  Changlang                            690.0     NaN    NaN  690001.0   
24465  Changlang                            690.0     NaN    NaN  690002.0   
24466  Changlang                            690.0     NaN    NaN  690002.0   
24467  Changlang                            690.0     NaN    NaN  690003.0   

       hh_multiplier  total_member  ...  vlg_govt_prim

In [47]:
# --- STEP 1: Fix "Object" columns ---

# Identify columns that are currently 'object' type (strings or mixed)
object_cols = master_data.select_dtypes(include=['object']).columns

for col in object_cols:
    # Check 1: Is the column entirely empty (all NaNs)?
    # Stata doesn't like "Object" NaNs. We must cast them to float (numeric missing).
    if master_data[col].isnull().all():
        master_data[col] = master_data[col].astype(float)
        continue

    # Check 2: Is this actually a numeric column disguised as object?
    # (e.g. contains "1", "2" and NaNs). Try to convert to numeric.
    try:
        # Try converting to numeric. If it fails (e.g. contains "Maharashtra"), 
        # it raises an error and goes to the 'except' block.
        # We use a temporary variable so we don't overwrite text columns with NaNs
        temp_col = pd.to_numeric(master_data[col])
        master_data[col] = temp_col
    except (ValueError, TypeError):
        # If conversion fails, it means it's a real string column (like state_name).
        # We must ensure all values are strings (replace NaNs with empty strings or keep as None)
        # Stata prefers pure strings.
        master_data[col] = master_data[col].astype(str).replace('nan', '')

# --- STEP 2: Enforce Numeric Types for known Categorical Vars ---
# This ensures variables like 'hh_type' don't get saved as strings "1", "2"

# List of columns that MUST be numeric
numeric_targets = [
    'hh_type', 'hh_electricity_conn', 'hh_toilet', 'school_type', 
    'read_code', 'math_code', 'child_age', 'total_member',
    'hh_motor_veh_2whl', 'hh_motor_veh_4whl'
]

for col in numeric_targets:
    if col in master_data.columns:
        master_data[col] = pd.to_numeric(master_data[col], errors='coerce')

# --- STEP 3: Truncate Column Names (Stata Limit) ---
# (Repeating this just to be safe, as it was in the previous traceback context)
clean_cols = {}
for col in master_data.columns:
    new_col = col.replace(" ", "_").replace("-", "_").replace(".", "")
    new_col = new_col[:32] # Stata strict limit
    clean_cols[col] = new_col

master_data.rename(columns=clean_cols, inplace=True)

# --- STEP 4: Save ---
print("Saving to Stata...")
try:
    master_data.to_stata("ASER_Panel_2007_2024.dta", version=118, write_index=False)
    print("Success! File saved as ASER_Panel_2007_2024.dta")
except Exception as e:
    print(f"Still failing. The specific error is:\n{e}")
    # If it fails, print the dtypes to debug
    print("\nCurrent Column Data Types:")
    print(master_data.dtypes)

Saving to Stata...
Success! File saved as ASER_Panel_2007_2024.dta
